# NB02 · Semana 4 — Extracción de URLhaus hacia la capa bronze

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 1 (E1)**

## Objetivo

Traer las URLs que distribuyen malware publicadas por **URLhaus (abuse.ch)** y guardarlas en la capa **bronze** del lake (Delta Lake en MinIO) cumpliendo tres condiciones:

| Requisito | Cómo se cumple acá |
|---|---|
| **Carga incremental** | El *watermark* es el `id` más alto que ya está en bronze. Solo se escriben URLs con `id` mayor. |
| **Idempotencia** | No se trae lo que ya está: la segunda corrida seguida encuentra 0 URLs nuevas y no escribe nada. Siempre `append`, nunca `overwrite`. |
| **Manejo de errores** | Se separan fallas **transitorias** (timeout, caída de red, HTTP 5xx o 429: se reintenta 3 veces con espera creciente) de **permanentes** (clave inválida, endpoint retirado, respuesta con otro formato: se corta sin reintentar y sin escribir). |

La Auth-Key de URLhaus se lee de la variable de entorno `URLHAUS_AUTH_KEY`: no está escrita en el notebook.

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BRONZE, STORAGE_OPTIONS, info
info()

# La Auth-Key vive en dags/.env (la escribe NB05). Si no está en el entorno del kernel, se toma de ahí.
if not os.environ.get("URLHAUS_AUTH_KEY"):
    _env = os.path.join(_d, "dags", ".env")
    if os.path.exists(_env):
        for _l in open(_env, encoding="utf-8"):
            if _l.startswith("URLHAUS_AUTH_KEY="):
                os.environ["URLHAUS_AUTH_KEY"] = _l.strip().split("=", 1)[1]
print("Auth-Key de URLhaus:", "cargada" if os.environ.get("URLHAUS_AUTH_KEY") else "FALTA")

In [ ]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

## 1. El *watermark*

Es el `id` más alto que ya está en bronze. URLhaus asigna ids crecientes: una URL nueva siempre tiene un id mayor que todas las anteriores. Se **deriva de los datos** (no se guarda en un archivo aparte), así que no puede desincronizarse de lo que realmente hay en bronze.

Si la tabla no existe, devuelve `None`: es la primera corrida. Cualquier otro error (MinIO caído, credenciales mal) **no** se toma como tabla vacía: se propaga, porque tratarlo como primera corrida haría una carga histórica encima de datos existentes.

In [ ]:
import pyarrow.compute as pc
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError


def leer_watermark():
    """Máximo id ya ingerido en bronze, o None si la tabla todavía no existe."""
    try:
        dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
    except TableNotFoundError:
        return None
    maximo = pc.max(dt.to_pyarrow_table(columns=["id"])["id"]).as_py()
    return None if maximo is None else int(maximo)


def contar_bronze():
    try:
        return DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS).to_pyarrow_table(columns=["id"]).num_rows
    except TableNotFoundError:
        return 0


print("watermark actual:", leer_watermark(), "· filas en bronze:", contar_bronze())

## 2. Pedidos a URLhaus con manejo de errores

`pedir()` es el único punto que habla con URLhaus. Verifica el código HTTP **antes** de leer el cuerpo y actúa según el tipo de falla:

| Qué pasa | Tipo | Qué hace |
|---|---|---|
| Timeout o error de conexión | transitoria | reintenta (hasta 3 intentos, esperando 5 s y 10 s) |
| HTTP 429 (demasiados pedidos) o 5xx | transitoria | reintenta igual |
| HTTP 401/403 (clave inválida), 404/410 (endpoint retirado), otro 4xx | permanente | corta en el acto: reintentar no lo arregla |
| 200 pero el cuerpo no es JSON, `query_status` no es `ok`, o faltan columnas | permanente | corta sin escribir nada en bronze |
| Filas con `id` o fecha ilegibles | dato malo | se descartan esas filas y se informa cuántas |

Los mensajes nunca muestran la Auth-Key (va dentro de la URL del export, así lo pide abuse.ch).

In [ ]:
import csv, io, time, zipfile
import pandas as pd
import pyarrow as pa
import requests

API_RECIENTES = "https://urlhaus-api.abuse.ch/v1/urls/recent/"
EXPORT_COMPLETO = "https://urlhaus-api.abuse.ch/v2/files/exports/{clave}/full.csv.zip"

COLUMNAS = ["id", "date_added", "url", "url_status", "last_online",
            "threat", "tags", "urlhaus_link", "reporter"]
# Mismo esquema que escribe extraer_bronze en el DAG: las dos ingestas conviven en la misma tabla
ESQUEMA = pa.schema([
    ("id", pa.int64()), ("date_added", pa.timestamp("us", tz="UTC")),
    ("url", pa.string()), ("url_status", pa.string()), ("last_online", pa.string()),
    ("threat", pa.string()), ("tags", pa.list_(pa.string())),
    ("urlhaus_link", pa.string()), ("reporter", pa.string()),
])


class FallaTransitoria(Exception):
    """URLhaus no respondió bien, pero puede andar en un rato."""


class FallaPermanente(Exception):
    """Reintentar no sirve: hay que cambiar algo (clave, endpoint, código)."""


def _clave():
    clave = os.environ.get("URLHAUS_AUTH_KEY")
    if not clave:
        raise FallaPermanente("Falta la variable de entorno URLHAUS_AUTH_KEY")
    return clave


def pedir(url, intentos=3, espera=5, timeout=30, stream=False):
    clave = _clave()
    visible = url.replace(clave, "<AUTH-KEY>")
    motivo = ""
    for intento in range(1, intentos + 1):
        try:
            res = requests.get(url, headers={"Auth-Key": clave}, timeout=timeout, stream=stream)
        except (requests.Timeout, requests.ConnectionError) as e:
            motivo = type(e).__name__          # el texto de la excepción incluye la URL: no se muestra
        else:
            if res.status_code == 200:
                return res
            if res.status_code == 429 or res.status_code >= 500:
                motivo = f"HTTP {res.status_code}"
            else:
                raise FallaPermanente(f"URLhaus respondió HTTP {res.status_code} en {visible}: "
                                      "clave inválida, endpoint retirado o pedido mal formado")
        if intento < intentos:
            print(f"  intento {intento}/{intentos} falló ({motivo}); reintento en {espera * intento} s")
            time.sleep(espera * intento)
    raise FallaTransitoria(f"URLhaus no respondió bien en {intentos} intentos ({motivo}) · {visible}")


def _tags(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    partes = valor if isinstance(valor, (list, tuple)) else str(valor).split(",")
    partes = [str(p).strip() for p in partes if str(p).strip()]
    return partes or None


def _texto(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    valor = str(valor).strip()
    return None if valor in ("", "None") else valor


def normalizar(df):
    """Valida columnas, tipa con el esquema de bronze, descarta filas ilegibles y deduplica por id."""
    faltan = set(COLUMNAS) - set(df.columns)
    if faltan:
        raise FallaPermanente(f"La respuesta no trae las columnas {sorted(faltan)}: cambió el formato")
    df = df[COLUMNAS].copy()
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    fechas = df["date_added"].astype(str).str.replace(" UTC", "", regex=False)
    df["date_added"] = pd.to_datetime(fechas, format="%Y-%m-%d %H:%M:%S", utc=True, errors="coerce")
    malas = df["id"].isna() | df["date_added"].isna()
    if malas.any():
        print(f"  {int(malas.sum())} filas descartadas por id o fecha ilegibles")
    df = df[~malas].copy()
    df["id"] = df["id"].astype("int64")
    df["tags"] = df["tags"].map(_tags)
    for col in ["url", "url_status", "last_online", "threat", "urlhaus_link", "reporter"]:
        df[col] = df[col].map(_texto)
    df = df.drop_duplicates(subset=["id"])
    return pa.Table.from_pandas(df, schema=ESQUEMA, preserve_index=False)

## 3. Extracción

Dos fuentes de URLhaus, con las mismas columnas:

* **Primera corrida (bronze vacío):** `full.csv.zip`, la base completa. Es el histórico.
* **Corridas siguientes:** la API `/v1/urls/recent/`, con las URLs de los últimos 3 días (máximo 1000).

**Límite de la fuente:** la API de URLhaus no acepta un parámetro "desde" (ni por fecha ni por id): siempre devuelve lo reciente. Por eso el watermark no viaja en el pedido sino que filtra la respuesta (`id > watermark`). Lo que sí se respeta es lo importante: **solo se escribe lo que no estaba**, y el pedido es chico (como mucho 1000 URLs), no el histórico entero.

El DAG de Airflow (NB05) resuelve el reproceso de fechas viejas con otro criterio (por intervalo de fecha), porque esta API no sirve para el pasado.

In [ ]:
def extraer_recientes(desde):
    """URLs de la API con id mayor al watermark."""
    res = pedir(API_RECIENTES)
    try:
        cuerpo = res.json()
    except ValueError:
        raise FallaPermanente("La API devolvió 200 pero el cuerpo no es JSON") from None
    estado = cuerpo.get("query_status")
    if estado == "no_results":
        return ESQUEMA.empty_table()
    if estado != "ok":
        raise FallaPermanente(f"La API respondió query_status={estado!r}")
    urls = cuerpo.get("urls") or []
    if not urls:
        return ESQUEMA.empty_table()
    df = pd.DataFrame(urls).rename(columns={"urlhaus_reference": "urlhaus_link"})
    if "last_online" not in df.columns:          # la API no lo trae; el CSV sí
        df["last_online"] = None
    tabla = normalizar(df)
    print(f"  la API devolvió {tabla.num_rows} URLs; con id > {desde}: ", end="")
    tabla = tabla.filter(pc.greater(tabla["id"], desde))
    print(tabla.num_rows)
    return tabla


def extraer_historico():
    """Base completa de URLhaus (solo para la primera corrida)."""
    res = pedir(EXPORT_COMPLETO.format(clave=_clave()), timeout=600, stream=True)
    datos = io.BytesIO()
    for bloque in res.iter_content(1 << 20):
        datos.write(bloque)
    if not zipfile.is_zipfile(datos):
        raise FallaPermanente("La descarga del histórico no es un ZIP (¿página de error?)")
    with zipfile.ZipFile(datos) as zf, zf.open(zf.namelist()[0]) as crudo:
        lineas = io.TextIOWrapper(crudo, encoding="utf-8", errors="replace", newline="")
        filas = [r[:len(COLUMNAS)] for r in csv.reader(l for l in lineas if not l.startswith("#"))
                 if len(r) >= len(COLUMNAS)]
    print(f"  histórico: {len(filas)} filas en el CSV")
    return normalizar(pd.DataFrame(filas, columns=COLUMNAS))

## 4. Escribir en bronze

Una sola función decide qué hacer según el watermark. Siempre escribe en modo `append` y solo filas nuevas: no pisa nada de lo que ya estaba.

In [ ]:
def ingerir():
    wm = leer_watermark()
    if wm is None:
        print("Bronze vacío: primera corrida, carga histórica completa")
        nuevos = extraer_historico()
    else:
        print(f"Watermark: id {wm}")
        nuevos = extraer_recientes(wm)
    if nuevos.num_rows == 0:
        print("Sin URLs nuevas: no se escribe nada en bronze")
        return 0
    write_deltalake(BRONZE, nuevos, mode="append", schema_mode="merge", storage_options=STORAGE_OPTIONS)
    ids = nuevos["id"]
    print(f"{nuevos.num_rows} URLs nuevas en bronze (ids {pc.min(ids).as_py()} a {pc.max(ids).as_py()})")
    return nuevos.num_rows

## 5. Demostración: incremental e idempotente

Dos corridas seguidas. La primera trae lo nuevo; la segunda arranca de un watermark ya actualizado y no encuentra nada. No es que deduplique: **no pide escribir lo que ya tiene**.

In [ ]:
antes = contar_bronze()
print("--- Corrida 1 ---")
n1 = ingerir()
print("\n--- Corrida 2 (inmediatamente después) ---")
n2 = ingerir()
despues = contar_bronze()

print(f"\nFilas en bronze: {antes} → {despues} (corrida 1: +{n1}, corrida 2: +{n2})")
assert n2 == 0, "La segunda corrida no debería traer nada"
assert despues == antes + n1, "Bronze creció distinto de lo que se escribió"
print("✅ Incremental e idempotente: la segunda corrida no agregó filas")

## 6. Demostración: una falla permanente no se reintenta

Un endpoint que no existe devuelve 404: `pedir()` corta en el primer intento, no escribe nada y lo informa. (Una falla transitoria se ve en el log como `intento 1/3 falló (...); reintento en 5 s`.)

In [ ]:
try:
    pedir("https://urlhaus-api.abuse.ch/v1/este-endpoint-no-existe/")
except FallaPermanente as e:
    print("FallaPermanente:", e)
print("Filas en bronze siguen siendo:", contar_bronze())